# T08 — scCODA: Compositional Analysis of Cell Type Fractions

**Tools:** `sccoda` (scCODA / tascCODA, Theis Lab)  
**Dataset:** PBMC 3k with simulated conditions  
**Papers:** [Büttner et al. 2021, Nature Communications](https://doi.org/10.1038/s41467-021-27150-6) · [Ostner et al. 2021, Science Advances](https://doi.org/10.1126/sciadv.abq1458)

---

## The compositional problem

After clustering, you have **cell type fractions** per sample. A naive question: does the fraction of CD14+ monocytes differ between healthy and disease?

But cell type fractions are **compositional data** — they sum to 1. This creates a fundamental statistical problem:
- If one cell type increases, others *must* decrease (even if biology didn't change them)
- Standard tests (t-test, Wilcoxon) ignore this constraint → **inflated false discovery rates**

## What scCODA does

scCODA models cell type counts per sample using a **Dirichlet-multinomial** model:
- Each sample is a draw from a Dirichlet distribution over cell types
- The Dirichlet concentrations are linked to covariates (condition, treatment) via log-linear regression
- A **reference cell type** is fixed (concentrations are relative)
- Bayesian inference provides credible intervals, not just point estimates

**tascCODA** (tree-aggregated scCODA) extends this to handle cell type hierarchies (e.g., T cell subtypes are nested under T cells).

```
  Sample i:
  [n_CD4T, n_CD8T, n_Mono, n_B, n_NK] ~ DirichletMultinomial(α)
  log(α_k) = β_0k + β_1k × condition
               ↑
         tested vs. 0 with FDR control
```

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

# sccoda is the package name on PyPI
try:
    import sccoda
    from sccoda.util import comp_ana as ca
    from sccoda.util import cell_composition_data as dat
    from sccoda.util import data_visualization as viz
    print(f'sccoda {sccoda.__version__}')
except ImportError:
    print('Install: pip install sccoda')
    raise

sc.settings.set_figure_params(dpi=80, facecolor='white')

## 1. Build a Compositional Dataset

scCODA works on a **sample × cell type count matrix** — not cell-level AnnData. We simulate multiple "samples" from PBMC 3k by bootstrapping and perturbing one cell type to create a disease vs. healthy contrast.

In [ ]:
# Load PBMC 3k with annotations
import os
if os.path.exists('../data/pbmc3k_processed.h5ad'):
    adata = sc.read_h5ad('../data/pbmc3k_processed.h5ad')
else:
    adata = sc.datasets.pbmc3k_processed()
    if 'louvain' in adata.obs and 'cell_type' not in adata.obs:
        adata.obs['cell_type'] = adata.obs['louvain']

print(adata.obs['cell_type'].value_counts())

In [ ]:
# Simulate 8 pseudo-samples: 4 "healthy", 4 "disease"
# Disease: CD14+ monocytes 2× enriched (simulated inflammation)
np.random.seed(42)

cell_types = adata.obs['cell_type'].unique().tolist()
n_samples = 8

# Base proportions from the real data
base_props = adata.obs['cell_type'].value_counts(normalize=True)

samples = []
for s in range(n_samples):
    condition = 'healthy' if s < 4 else 'disease'
    n_cells = np.random.randint(200, 400)  # variable sample size
    
    # In disease: boost monocyte fraction 2×
    props = base_props.copy()
    if condition == 'disease' and 'CD14+ Mono' in props.index:
        props['CD14+ Mono'] *= 2.0
        props = props / props.sum()  # renormalize
    
    counts = np.random.multinomial(n_cells, props[cell_types].values)
    samples.append({'sample': f'S{s+1}', 'condition': condition, **dict(zip(cell_types, counts))})

counts_df = pd.DataFrame(samples).set_index('sample')
print(counts_df)

In [ ]:
# Create scCODA compositional data object
# covariate_columns: which columns in the DataFrame are metadata (not counts)
data = dat.from_pandas(counts_df, covariate_columns=['condition'])
print(data)

## 2. Exploratory Visualization

In [ ]:
# Stacked bar chart of cell type proportions per sample
viz.stacked_barplot(data, feature_name='condition', figsize=(8, 4))

In [ ]:
# Box plots of fractions per cell type, split by condition
viz.boxplots(
    data,
    feature_name='condition',
    add_dots=True,
    figsize=(12, 4),
)

## 3. scCODA Model: Bayesian Compositional Regression

The key choices:
- **formula**: which covariate(s) to test (R-style formula)
- **reference_cell_type**: the "baseline" cell type. All effects are relative to it. 
  Choose one that is unlikely to change (e.g., a house-keeping population).

In [ ]:
# Set up the model
# formula: test for 'condition' effect
# reference_cell_type: 'CD4+ T' as baseline (typically T cells in PBMC)
model = ca.CompositionalAnalysis(
    data,
    formula='condition',
    reference_cell_type='CD4+ T',
)
print(model)

In [ ]:
# MCMC sampling (No-U-Turn Sampler)
# num_results: MCMC samples to draw; 5000 for production
sim_results = model.sample_hmc(
    num_results=2000,
    num_burnin=500,
    num_adapt_steps=500,
)
print(sim_results)

## 4. Results: Which Cell Types Changed?

scCODA uses **spike-and-slab** priors: each effect β_k has a binary indicator (in model / out of model). The **Final Parameter** column in the summary shows whether a cell type is credibly changed.

In [ ]:
# Summary of credible effects
# 'Final Parameter' != 0 → credible change in that cell type
sim_results.summary()

In [ ]:
# Credible interval plot: which cell types are significantly altered?
viz.effects_barplot(
    sim_results,
    covariates=['condition[T.healthy]'],
    figsize=(7, 4),
)

In [ ]:
# Posterior distribution of effects
viz.plot_posterior_predictive(
    sim_results,
    feature_name='condition',
    figsize=(10, 4),
)

## 5. tascCODA: Tree-Aggregated Extension

tascCODA uses a hierarchical tree of cell types to improve power for detecting changes at higher levels of the hierarchy (e.g., "all T cells" change, not just one subtype).

In [ ]:
# tascCODA requires a cell type tree (e.g., from a cell ontology)
# Here we define a simple manual tree
try:
    from sccoda.util import tasccoda as tasccoda_util
    
    # Define a simple 2-level hierarchy
    # tree: dict of {parent: [children]}
    tree = {
        'All': ['Lymphoid', 'Myeloid', 'Other'],
        'Lymphoid': ['CD4+ T', 'CD8+ T', 'NK', 'B cell'],
        'Myeloid': ['CD14+ Mono', 'DC'],
        'Other': ['Mega'],
    }
    
    tasc_model = tasccoda_util.TreeAggregatedCompositionalAnalysis(
        data,
        formula='condition',
        reference_cell_type='CD4+ T',
        tree=tree,
    )
    tasc_results = tasc_model.sample_hmc(num_results=2000, num_burnin=500)
    tasc_results.summary()
except Exception as e:
    print(f'tascCODA: {e}')
    print('See: https://sccoda.readthedocs.io/en/latest/tasccoda.html')

---
## Summary

```python
# scCODA workflow
data = dat.from_pandas(counts_df, covariate_columns=['condition'])
model = ca.CompositionalAnalysis(data, formula='condition',
                                  reference_cell_type='CD4+ T')
results = model.sample_hmc(num_results=5000)
results.summary()   # Final Parameter column: 0 = no change, non-zero = credible change
```

**Key insight:** scCODA is conservative — it requires strong evidence to declare a change. This is by design, since compositional data makes it easy to get spurious associations. If you need higher sensitivity, try **Milo** (notebook 13 in perturb_seq/) which tests at the kNN neighborhood level.

| Model | Best for |
|-------|---------|
| `scCODA` | Cluster-level fractions, small number of cell types |
| `tascCODA` | Hierarchical cell type ontologies |
| `Milo` | Sub-cluster / continuous gradients, no need to commit to clusters |

**Done with the Theis ecosystem series!** You've now covered:
- **T00** AnnData + Scanpy (+ doublet detection, ambient RNA)
- **T01** scvi-tools
- **T02** Squidpy
- **T03** CellRank 2
- **T04** Muon + MOFA+
- **T05** LIANA (cell-cell communication)
- **T06** cell2location (spatial deconvolution)
- **T07** Moscot (optimal transport)
- **T08** scCODA (compositional analysis)